# Selective State Space Model (COMING SOON)

In this section, we forecast EEG activity using a Selective State Space Model (S6), the core operator underlying the Mamba architecture. State Space Models (SSMs) cast a time series as a continuous dynamical system whose hidden state evolves according to learned differential equations, then discretize those equations to efficiently process long sequences. S6 extends this framework with input-dependent, time-varying parameters that allow the model to selectively retain or discard information as it flows through the state. This positions S6 as a compromise between transformers and recurrent networks: transformers attend to every past time point simultaneously but at quadratic memory cost, while recurrent networks like GRUs run in constant memory but struggle to carry information across many steps. S6 inherits the linear-time inference of recurrent networks while matching the long-range selectivity of attention, making it well suited for the slow drifts and fast oscillatory bursts characteristic of EEG.

## Background

### State Space Models

A State Space Model (SSM) describes how a system evolves over time through a hidden state vector $h(t) \in \mathbb{R}^N$ that summarizes everything the model has observed up to time $t$. Given a scalar input $x(t) \in \mathbb{R}$ and a scalar output $y(t) \in \mathbb{R}$, the continuous-time SSM is defined by two equations:

$$
\begin{align}
h'(t) &= \mathbf{A}h(t) + \mathbf{B}x(t) \tag{1} \\
y(t) &= \mathbf{C}h(t) \tag{2}
\end{align}
$$

Equation (1) is a first-order linear ODE: the rate of change of the hidden state is a linear combination of the current state (governed by the state transition matrix $\mathbf{A} \in \mathbb{R}^{N \times N}$) and the new input (scaled by the input projection $\mathbf{B} \in \mathbb{R}^{N \times 1}$). Equation (2) reads out a prediction by projecting the hidden state through the output matrix $\mathbf{C} \in \mathbb{R}^{1 \times N}$. The matrices $(\mathbf{A}, \mathbf{B}, \mathbf{C})$ are learned parameters shared across all time steps.

Because real sequences are discrete (EEG is sampled at a fixed rate $F_s$), we must discretize Equations (1-2) using a step size $\Delta = 1/F_s$. Applying the zero-order hold (ZOH) approximation yields:

$$
\begin{align}
\bar{\mathbf{A}} &= e^{\Delta \mathbf{A}} \tag{3} \\
\bar{\mathbf{B}} &= (\Delta \mathbf{A})^{-1}(e^{\Delta \mathbf{A}} - \mathbf{I}) \cdot \Delta \mathbf{B} \tag{4}
\end{align}
$$

where $\bar{\mathbf{A}}$ and $\bar{\mathbf{B}}$ are the discretized counterparts. The recurrence then becomes:

$$
\begin{align}
h_t &= \bar{\mathbf{A}} h_{t-1} + \bar{\mathbf{B}} x_t \tag{5} \\
y_t &= \mathbf{C} h_t \tag{6}
\end{align}
$$

This is structurally identical to a linear RNN: the hidden state $h_t$ is updated at each time step and passed forward. The SSM therefore inherits the core strength of recurrent networks, namely the ability to process sequences of arbitrary length in $\mathcal{O}(L)$ time and constant memory.

### Shortcomings of Linear SSMs

Despite their elegance, linear SSMs with fixed parameters face two fundamental limitations:

- **Gradient instability.** Training requires backpropagating through every time step of the recurrence in Equation (5). When $\bar{\mathbf{A}}$ has eigenvalues with magnitude greater than one, the gradient signal grows exponentially with sequence length (exploding gradients); when eigenvalues fall below one, the gradient vanishes and the model cannot learn dependencies spanning more than a handful of steps. This problem is well-documented in vanilla RNNs and applies equally to SSMs trained on long sequences.

- **Content-agnostic transitions.** The matrices $(\mathbf{A}, \mathbf{B}, \mathbf{C})$ are global constants: every input token at every time step is processed identically regardless of what it contains. Formally, the map $x_t \mapsto h_t$ is linear and shift-invariant. For a non-stationary signal like EEG, this is restrictive. The brain transitions between distinct functional states (e.g., a pre-stimulus baseline versus a sharp evoked response) suggesting that a model with temporally adaptive weights is required to capture multipe regimes.

### HiPPO: Initializing the State Matrix

Before discussing how S6 addresses the limitations above, it is worth understanding how the state matrix $\mathbf{A}$ is initialized. Gu et al. (2020) introduced the **Hi**gh-order **P**olynomial **P**rojection **O**perator (HiPPO) as a principled initialization for $\mathbf{A}$ that allows the hidden state $h(t)$ to function as a compressed memory of the input history.

The core idea is to define $h(t)$ as the coefficients of a polynomial basis (specifically Legendre polynomials) that optimally approximates the function $x(s)$ for all $s \leq t$ under a sliding exponential window:

$$
x(s) \approx \sum_{n=0}^{N-1} h_n(t) \, p_n(s), \quad s \leq t \tag{7}
$$

where $p_n(s)$ are the basis polynomials and $h_n(t)$ are their time-varying coefficients stored in the hidden state. Each new input $x_t$ causes the coefficient vector to be updated so that the approximation remains optimal. Critically, this update can be written exactly as a linear recurrence of the form in Equation (5), with the transition matrix $\mathbf{A}$ taking a specific closed-form structure derived from the polynomial basis:

$$
A_{nk} = -\begin{cases} (2n+1)^{1/2}(2k+1)^{1/2} & \text{if } n > k \\ n+1 & \text{if } n = k \\ 0 & \text{if } n < k \end{cases} \tag{8}
$$

This lower-triangular structure ensures that lower-order coefficients (capturing coarse, long-range trends) are always informed by higher-order ones (capturing fine, short-range detail), giving the model a principled inductive bias toward retaining long-range context. Using HiPPO to initialize $\mathbf{A}$ largely resolves the gradient instability problem: eigenvalues of the HiPPO matrix lie on the imaginary axis, producing oscillatory rather than growing or decaying dynamics during the forward pass.

### S6: The Selective Scan

The S6 layer (Gu & Dao, 2023) addresses the content-agnostic limitation by making the parameters $\mathbf{B}$, $\mathbf{C}$, and the discretization step $\Delta$ functions of the input at each time step. Instead of fixed global matrices, the model learns projection layers that map each $x_t$ to its own parameter set:

$$
\begin{align}
\mathbf{B}_t &= \text{Linear}_B(x_t) \in \mathbb{R}^{N} \tag{9} \\
\mathbf{C}_t &= \text{Linear}_C(x_t) \in \mathbb{R}^{N} \tag{10} \\
\Delta_t &= \text{Softplus}(\text{Linear}_\Delta(x_t)) \in \mathbb{R}^{D} \tag{11}
\end{align}
$$

where $D$ is the model dimension. Each $\Delta_t$ is then used to discretize $\mathbf{A}$ via Equations (3-4), yielding a time-varying $\bar{\mathbf{A}}_t$. The recurrence becomes:

$$
\begin{align}
h_t &= \bar{\mathbf{A}}_t \, h_{t-1} + \bar{\mathbf{B}}_t \, x_t \tag{12} \\
y_t &= \mathbf{C}_t \, h_t \tag{13}
\end{align}
$$

The key intuition behind this *selection mechanism* is that $\Delta_t$ controls how much the model focuses on the current input versus carrying forward prior context. A large $\Delta_t$ causes $\bar{\mathbf{A}}_t \to 0$ and $\bar{\mathbf{B}}_t \to \mathbf{B}$, so the state resets and attends to the present sample. A small $\Delta_t$ keeps $\bar{\mathbf{A}}_t \approx \mathbf{I}$, letting the state coast over an uninformative region without overwriting prior memory. For EEG, this means the model can snap to attention during a sharp evoked potential and compress featureless inter-stimulus intervals, all within a single recurrent pass. Crucially, $\mathbf{A}$ itself remains fixed (initialized via HiPPO) and is not made input-dependent, preserving the stable spectral structure that enables long-range memory.

### Hardware-Aware Implementation

One practical challenge with the selective scan is that input-dependent parameters break the fixed convolutional kernel that earlier SSMs (e.g., S4) used for efficient parallelization. Gu & Dao (2023) address this with a hardware-aware algorithm that fuses the scan kernel and keeps intermediate states in fast SRAM (on-chip GPU memory) rather than repeatedly reading and writing to slower VRAM. This dramatically reduces memory bandwidth overhead and allows Mamba to run faster than comparably sized Transformers on long sequences without sacrificing the expressiveness gained by selectivity. A detailed treatment of this hardware optimization is beyond the scope of this tutorial, but more info can be found at [link].

### Connection to Dynamic Mode Decomposition

S6 and DMD share a common theoretical ancestor: both are grounded in the idea that a dynamical system's evolution can be approximated by a linear operator acting on a latent state. In DMD, this operator is the matrix $\mathbf{A}$ estimated from data snapshots, whose eigendecomposition yields interpretable spatial modes and frequencies. In S6, the same role is played by the discretized $\bar{\mathbf{A}}_t$, which propagates the hidden state forward at each time step.

The key difference lies in how each method handles the operator and what it represents. DMD fits a single global $\mathbf{A}$ to the entire training window, then forecasts by extrapolating the learned eigenstructure. It is a closed-form, non-parametric method with no gradient-based optimization. S6, by contrast, learns $\mathbf{A}$ end-to-end through backpropagation, initializes it with the HiPPO structure rather than data-driven eigendecomposition, and crucially allows the effective transition operator to vary with the input via $\Delta_t$. Where DMD assumes the underlying dynamics are stationary, S6 makes no such assumption and can adapt its state transitions to non-stationary shifts in neural activity. In this sense, S6 can be thought of as a learned, input-conditional generalization of the linear dynamical system that DMD fits explicitly.

## Implementing S6

Now lets put the theory into practice. Rather than relying on an external library, we will build the S6 selective scan and the surrounding Mamba block from scratch in PyTorch.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import math
from tqdm import trange
import matplotlib.pyplot as plt

First, let's load the SSVEP dataset from Gu et al. (2024). As in the GRU and Transformer tutorials, we use every trial across all stimulation frequencies and both blocks to maximise the amount of training data.

In [ ]:
# load single subject from Gu et al., 2024 SSVEP experiment
# Note: data has been downsampled to 250 Hz
data: np.ndarray = np.load('../dataset/Data/data_s1_64_down.npy')

# block x stimulation frequency x time x channels x conditions (low and high luminance ratios)
nBlocks, nFreqs, nTime, nChans, nCons = data.shape

# select high luminance condition and collapse blocks x stim frequencies into a single trial axis
con_idx: int = 1
X: np.ndarray = data[:, :, :, :, con_idx]
X = X.reshape(-1, *X.shape[2:])

# transpose to (trials, channels, time)
X = np.swapaxes(X, 1, 2)

# SSVEP responses are strongest over posterior occipital electrodes
ssvep_chan_names: list[str] = ['Pz', 'PO5', 'PO3', 'POz', 'PO4', 'PO6', 'O1', 'Oz', 'O2']
ssvep_chans_idx: list[int]  = [48, 54, 55, 56, 57, 58, 61, 62, 63]
ssvep_chans_dict: dict[str, int] = dict(zip(ssvep_chan_names, ssvep_chans_idx))

# time vector: 5 s stimulus + 0.14 s post-offset, sampled at 250 Hz
Fs: float = 250.
t: np.ndarray = np.arange(0, 5.14, 1 / Fs) * 1000  # ms

In [ ]:
class customDataSet(Dataset):
    def __init__(self, data: torch.Tensor):
        self.data: torch.Tensor = data

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> torch.Tensor:
        # return a single trial of shape (channels, time)
        return self.data[idx, :]

### S6 Selective Scan

The `S6` module implements Equations (9–13) from the Background section. For each input token $x_t$, three small linear layers produce the time-varying parameters $\mathbf{B}_t$, $\mathbf{C}_t$, and $\Delta_t$. The step size $\Delta_t$ discretizes the fixed state matrix $\mathbf{A}$ via the zero-order hold approximation, yielding a per-step $\bar{\mathbf{A}}_t$ and $\bar{\mathbf{B}}_t$, and the hidden state is then updated sequentially and read out through $\mathbf{C}_t$. $\mathbf{A}$ is stored as its negative logarithm (`A_log`) to keep the diagonal of $\bar{\mathbf{A}}_t$ in $(0, 1)$, preventing divergence, and is initialized with the HiPPO spectrum.

In [ ]:
def hippo_init(N: int) -> torch.Tensor:
    """
    Constructs the HiPPO-LegS state matrix A of shape (N, N).
    Entries follow Equation (8): lower-triangular with a specific closed form.
    Only the diagonal is used in the diagonal SSM variant (S4D/Mamba), so we
    return the diagonal as the initialization vector for A_log.
    """
    # build the full HiPPO matrix then extract its diagonal
    A: np.ndarray = np.zeros((N, N))
    for n in range(N):
        for k in range(N):
            if n > k:
                A[n, k] = -((2 * n + 1) ** 0.5) * ((2 * k + 1) ** 0.5)
            elif n == k:
                A[n, k] = -(n + 1)
    # diagonal SSM uses only the diagonal entries; negate for log-parameterization
    diag: np.ndarray = np.diag(A)          # all negative: -(n+1)
    return torch.tensor(-diag, dtype=torch.float32)  # store positive values in A_log


class S6(nn.Module):
    """
    Selective State Space (S6) layer.

    Args:
        d_model:    Input/output feature dimension D.
        d_state:    Hidden state dimension N.
        dt_rank:    Rank of the Delta projection (typically ceil(D / 16)).
    """

    def __init__(self, d_model: int, d_state: int = 16, dt_rank: int | None = None):
        super().__init__()
        self.d_model: int  = d_model
        self.d_state: int  = d_state
        self.dt_rank: int  = dt_rank or math.ceil(d_model / 16)

        # A_log: (D, N) — log-parameterized diagonal state matrix, initialized with HiPPO
        # A is shared across time steps; only B, C, Delta are input-dependent
        A_init: torch.Tensor = hippo_init(d_state).unsqueeze(0).expand(d_model, -1).clone()
        self.A_log = nn.Parameter(torch.log(A_init + 1e-4))   # stored as log for stability

        # D (skip connection scalar, one per channel)
        self.D = nn.Parameter(torch.ones(d_model))

        # Input-dependent projections: map x_t -> (Delta_rank, B, C)
        self.x_proj = nn.Linear(d_model, self.dt_rank + 2 * d_state, bias=False)

        # Delta projection: low-rank -> full D dimension, then softplus for positivity
        self.dt_proj = nn.Linear(self.dt_rank, d_model, bias=True)

    def selective_scan(
        self,
        u: torch.Tensor,       # (B, L, D)  — input sequence
        delta: torch.Tensor,   # (B, L, D)  — discretization step
        A: torch.Tensor,       # (D, N)     — continuous state matrix (recovered from A_log)
        B: torch.Tensor,       # (B, L, N)  — input-dependent input projection
        C: torch.Tensor,       # (B, L, N)  — input-dependent output projection
    ) -> torch.Tensor:
        """
        Runs the discrete SSM recurrence (Equations 12-13) sequentially over time.
        Returns y of shape (B, L, D).
        """
        batch_size: int
        seq_len: int
        d_in: int
        batch_size, seq_len, d_in = u.shape
        n: int = A.shape[1]

        # ZOH discretization: dA_t = exp(Delta_t * A),  dB_t = Delta_t * B_t
        # delta: (B, L, D) -> unsqueeze N dim -> (B, L, D, N)
        dA: torch.Tensor = torch.exp(delta.unsqueeze(-1) * A)           # (B, L, D, N)
        dB: torch.Tensor = delta.unsqueeze(-1) * B.unsqueeze(2)         # (B, L, D, N)

        # initialise hidden state h to zero
        h: torch.Tensor = torch.zeros(batch_size, d_in, n, device=u.device, dtype=u.dtype)
        ys: list[torch.Tensor] = []

        for t_step in range(seq_len):
            # update hidden state: h_t = dA_t * h_{t-1} + dB_t * x_t
            h = dA[:, t_step] * h + dB[:, t_step] * u[:, t_step].unsqueeze(-1)
            # read out: y_t = C_t @ h_t  (sum over state dimension N)
            y_t: torch.Tensor = (h * C[:, t_step].unsqueeze(1)).sum(dim=-1)  # (B, D)
            ys.append(y_t)

        return torch.stack(ys, dim=1)   # (B, L, D)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, L, D)
        Returns:
            (B, L, D)
        """
        # recover A from its log-parameterization (always positive diagonal)
        A: torch.Tensor = -torch.exp(self.A_log)           # (D, N)

        # single projection produces Delta (low-rank), B, and C from each token
        x_dbl: torch.Tensor = self.x_proj(x)               # (B, L, dt_rank + 2*N)
        dt_raw, B, C = x_dbl.split([self.dt_rank, self.d_state, self.d_state], dim=-1)

        # project Delta to full model dimension and apply softplus for positivity
        delta: torch.Tensor = F.softplus(self.dt_proj(dt_raw))  # (B, L, D)

        # run the selective scan
        y: torch.Tensor = self.selective_scan(x, delta, A, B, C)

        # add skip connection scaled by learnable D
        return y + x * self.D

### Mamba Block

The `S6` layer sits inside a `MambaBlock` that adds the gating and normalisation structure from the original Mamba paper. The block applies a `LayerNorm`, then splits the input into two branches projected to an expanded inner dimension $E = \text{expand} \times D$. One branch passes through a short causal depthwise convolution followed by the S6 scan; the other is activated with `SiLU` and multiplied element-wise against the scan output, acting as a gate that suppresses uninformative transitions. The gated result is projected back to $D$ and added to the residual. The causal conv gives each position a local receptive field before the global recurrence begins, improving early-layer gradient flow.

In [ ]:
class MambaBlock(nn.Module):
    """
    Single Mamba residual block wrapping an S6 selective scan.

    Args:
        d_model:    Model dimension D.
        d_state:    SSM hidden state dimension N.
        expand:     Inner expansion factor E = expand * D.
        conv_size:  Kernel size of the causal depthwise convolution.
        dropout:    Dropout probability applied after the output projection.
    """

    def __init__(
        self,
        d_model: int,
        d_state: int  = 16,
        expand: int   = 2,
        conv_size: int = 4,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.d_inner: int = expand * d_model

        # pre-norm
        self.norm = nn.LayerNorm(d_model)

        # two parallel input projections to the expanded dimension
        self.in_proj  = nn.Linear(d_model, self.d_inner * 2, bias=False)
        self.out_proj = nn.Linear(self.d_inner, d_model, bias=False)

        # causal depthwise conv: groups=d_inner processes each channel independently
        # padding = conv_size - 1 ensures causality after trimming
        self.conv1d = nn.Conv1d(
            in_channels=self.d_inner,
            out_channels=self.d_inner,
            kernel_size=conv_size,
            padding=conv_size - 1,
            groups=self.d_inner,
            bias=True,
        )

        # S6 selective scan operating in the expanded dimension
        self.ssm = S6(d_model=self.d_inner, d_state=d_state)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, L, D)
        Returns:
            (B, L, D)
        """
        residual: torch.Tensor = x
        x = self.norm(x)

        # project to two branches of size (B, L, d_inner) each
        xz: torch.Tensor = self.in_proj(x)
        x_branch, z = xz.chunk(2, dim=-1)           # (B, L, d_inner) each

        # causal depthwise conv along the time axis, then trim the non-causal padding
        x_conv: torch.Tensor = self.conv1d(x_branch.transpose(1, 2))
        x_conv = x_conv[:, :, :x_branch.size(1)]    # trim future padding
        x_conv = F.silu(x_conv).transpose(1, 2)     # (B, L, d_inner)

        # selective scan over the convolved features
        y: torch.Tensor = self.ssm(x_conv)

        # multiplicative gate: suppress uninformative state transitions
        y = y * F.silu(z)

        # project back to model dimension and add residual
        return residual + self.dropout(self.out_proj(y))

### MambaForecaster

`MambaForecaster` assembles the full model. A linear input projection maps the raw EEG channel values at each time step into the model's embedding dimension. The signal then flows through a stack of `MambaBlock` layers, each of which maintains its own hidden state that evolves selectively based on the input. After the final block, a `LayerNorm` stabilises the representation and the last time step's embedding is passed to a linear output head that directly predicts all `pred_len` forecast steps across all channels simultaneously.

In [ ]:
class MambaForecaster(nn.Module):
    """
    Stacked Mamba (S6) model for multivariate EEG forecasting.

    Args:
        n_channels:  Number of EEG channels (input and output features).
        d_model:     Embedding dimension carried through the Mamba blocks.
        d_state:     SSM hidden state dimension N per block.
        n_layers:    Number of stacked MambaBlocks.
        expand:      Inner expansion factor inside each MambaBlock.
        pred_len:    Number of future time steps to forecast.
        dropout:     Dropout probability applied inside each block.
    """

    def __init__(
        self,
        n_channels: int,
        d_model: int   = 64,
        d_state: int   = 16,
        n_layers: int  = 4,
        expand: int    = 2,
        pred_len: int  = 64,
        dropout: float = 0.05,
    ):
        super().__init__()
        self.pred_len: int   = pred_len
        self.n_channels: int = n_channels

        # project raw EEG channel values into the embedding space
        self.input_proj = nn.Linear(n_channels, d_model)

        # stack of Mamba residual blocks
        self.layers = nn.ModuleList([
            MambaBlock(d_model=d_model, d_state=d_state, expand=expand, dropout=dropout)
            for _ in range(n_layers)
        ])

        # final normalisation before the forecast head
        self.norm = nn.LayerNorm(d_model)

        # direct multi-step head: last hidden state -> all pred_len steps x channels
        self.output_proj = nn.Linear(d_model, pred_len * n_channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (B, T, C) — input EEG window, channels last.
        Returns:
            (B, pred_len, C) — forecasted values.
        """
        # embed each time step from channel space into model dimension
        z: torch.Tensor = self.input_proj(x)          # (B, T, d_model)

        # pass through each Mamba block sequentially
        for layer in self.layers:
            z = layer(z)

        # take the final time step as the sequence summary
        z_last: torch.Tensor = self.norm(z[:, -1, :])  # (B, d_model)

        # project to all forecast steps and reshape
        out: torch.Tensor = self.output_proj(z_last)   # (B, pred_len * n_channels)
        return out.view(x.size(0), self.pred_len, self.n_channels)

With the model defined, let's split the data into training, validation, and test sets, normalise using training statistics, and build the data loaders. The splits and normalisation procedure are identical to the GRU and Transformer tutorials.

In [ ]:
nTrials: int = X.shape[0]
nTestTrials: int = math.ceil(nTrials * 0.2)   # 20% held out for test
shuffIdx: np.ndarray = np.random.permutation(nTrials)

X_test: np.ndarray      = X[shuffIdx[-nTestTrials:], :]
X_train_val: np.ndarray = X[shuffIdx[:-nTestTrials], :]

nValidTrials: int = math.ceil(X_train_val.shape[0] * 0.2)  # 20% of remainder for validation
X_val: np.ndarray   = X_train_val[-nValidTrials:, :]
X_train: np.ndarray = X_train_val[:-nValidTrials, :]

In [ ]:
X_train_mu: float = X_train.mean()
X_train_sd: float = X_train.std()

# normalise using training statistics only to prevent data leakage
X_train = (X_train - X_train_mu) / X_train_sd
X_val   = (X_val   - X_train_mu) / X_train_sd
X_test  = (X_test  - X_train_mu) / X_train_sd

In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float32)
X_val   = torch.tensor(X_val,   dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)

train_dataset = customDataSet(X_train)
val_dataset   = customDataSet(X_val)
test_dataset  = customDataSet(X_test)

In [ ]:
BATCH_SIZE:  int   = 64
LEARN_RATE:  float = 1e-3
N_EPOCHS:    int   = 150
HORIZON_MS:  int   = 1000
HORIZON:     int   = abs(t - HORIZON_MS).argmin().item() + 1  # samples in the forecast window
SEQ_LENGTH:  int   = nTime - HORIZON
UPDATE_FREQ: int   = 50   # print validation loss every N epochs

trainDL = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
valDL   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
testDL  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

Now we instantiate the model. Like the GRU, the `MambaForecaster` expects channels-last input of shape `(B, T, C)`, so we transpose each batch before passing it through the model and transpose predictions back to `(B, C, pred_len)` to match the target.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = MambaForecaster(
    n_channels=nChans,   # one input feature per EEG channel
    d_model=64,          # embedding dimension inside each Mamba block
    d_state=16,          # SSM hidden state size N
    n_layers=4,          # number of stacked MambaBlocks
    expand=2,            # inner expansion factor (d_inner = 2 * d_model)
    pred_len=HORIZON,    # forecast horizon (1000 ms)
    dropout=0.05,
).to(device)

optimizer = optim.NAdam(model.parameters(), lr=LEARN_RATE)
criterion = nn.MSELoss()

In [ ]:
train_loss: list[float] = []
val_loss:   list[float] = []

for iEpoch in trange(N_EPOCHS):
    model.train()
    epoch_train_loss: float = 0.0

    for train_batch in trainDL:
        # batches are (B, C, T); split chronologically into input and target windows
        input_sequence:  torch.Tensor = train_batch[:, :, :-HORIZON].to(device)
        target_sequence: torch.Tensor = train_batch[:, :, -HORIZON:].to(device)

        # model expects channels-last (B, T, C); predictions are (B, pred_len, C)
        batch_predictions: torch.Tensor = model(input_sequence.transpose(1, 2)).transpose(1, 2)
        batch_loss: torch.Tensor = criterion(batch_predictions, target_sequence)

        optimizer.zero_grad()
        batch_loss.backward()
        optimizer.step()

        epoch_train_loss += batch_loss.item()

    train_loss.append(epoch_train_loss / len(trainDL))

    if (iEpoch + 1) % UPDATE_FREQ == 0:
        model.eval()
        epoch_val_loss: float = 0.0

        with torch.no_grad():
            for val_batch in valDL:
                input_sequence  = val_batch[:, :, :-HORIZON].to(device)
                target_sequence = val_batch[:, :, -HORIZON:].to(device)

                batch_predictions = model(input_sequence.transpose(1, 2)).transpose(1, 2)
                batch_loss = criterion(batch_predictions, target_sequence)
                epoch_val_loss += batch_loss.item()

        val_loss.append(epoch_val_loss / len(valDL))
        print(f"Epoch [{iEpoch + 1}/{N_EPOCHS}] | Train Loss: {epoch_train_loss/len(trainDL):.3e} | Val Loss: {epoch_val_loss/len(valDL):.3e}")

Let's inspect the training and validation loss curves to check for overfitting before evaluating on the test set.

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(np.arange(N_EPOCHS) + 1, train_loss, marker='o', label='Training')
plt.plot(
    [x for x in np.arange(N_EPOCHS) + 1 if not x % UPDATE_FREQ],
    val_loss,
    marker='o',
    label='Validation'
)
plt.title('Training Performance')
plt.xlabel('Epoch')
plt.ylabel('MSE')
plt.legend(frameon=False)
plt.show()

Now let's run inference on the held-out test set, denormalise the predictions back to microvolts, and compute the test RMSE.

In [ ]:
model.eval()
with torch.no_grad():
    input_sequence:  torch.Tensor = X_test[:, :, :-HORIZON].to(device)
    target_sequence: torch.Tensor = X_test[:, :, -HORIZON:].to(device)

    predictions: torch.Tensor = model(input_sequence.transpose(1, 2)).transpose(1, 2)

    # denormalise both predictions and targets back to µV
    target_denorm: np.ndarray = ((target_sequence * X_train_sd) + X_train_mu).cpu().numpy()
    predictions:   np.ndarray = ((predictions     * X_train_sd) + X_train_mu).cpu().numpy()

test_rmse: float = np.sqrt(((target_denorm - predictions) ** 2).mean())
print(f'Test RMSE: {test_rmse:.3e}')

Let's visualise the single-trial forecast for four representative posterior electrodes. We include 100 samples of context before the horizon boundary to make the transition visible.

In [ ]:
X_test_denorm: np.ndarray = ((X_test * X_train_sd) + X_train_mu).cpu().numpy()

fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(10, 6), dpi=100, sharex=True, sharey=True)
trial_idx: int = 0
time_plot_buffer: int = 100   # samples of context shown before the horizon
axs = axs.ravel()

for i, c in enumerate(['Oz', 'POz', 'O1', 'O2']):
    axs[i].plot(
        t[-(HORIZON + time_plot_buffer):],
        X_test_denorm[trial_idx, ssvep_chans_dict[c], -(HORIZON + time_plot_buffer):],
        color='k', alpha=0.4, label='Truth'
    )
    axs[i].plot(t[-HORIZON:], predictions[trial_idx, ssvep_chans_dict[c], :], color='steelblue', alpha=0.8, label='Forecast')
    axs[i].axvline(t[-HORIZON], color='black', linestyle=':', label='Horizon Start')
    axs[i].set_title(c)

    if i > 1:
        axs[i].set_xlabel('Time (ms)')
    if i in [0, 2]:
        axs[i].set_ylabel(r'Voltage ($\mu V$)')
    if i == 3:
        axs[i].legend(frameon=False)

plt.suptitle(f'S6 / Mamba Forecast\n(Horizon={HORIZON_MS} ms  Trial={trial_idx + 1})')
plt.tight_layout()
plt.show()

Single-trial forecasts are noisy by nature. Averaging across all test trials cancels trial-specific fluctuations and reveals the mean temporal trajectory the model has learned. The shaded region shows ±1 SEM across trials.

In [ ]:
average_preds: np.ndarray = predictions.mean(axis=0)      # (C, HORIZON)
average_truth: np.ndarray = X_test_denorm.mean(axis=0)    # (C, T)
preds_sem:     np.ndarray = predictions.std(axis=0) / np.sqrt(predictions.shape[0])

fig, axs = plt.subplots(nrows=2, ncols=2, figsize=(10, 6), dpi=100, sharex=True, sharey=True)
time_plot_buffer = 100
axs = axs.ravel()

for i, c in enumerate(['Oz', 'POz', 'O1', 'O2']):
    axs[i].plot(
        t[-(HORIZON + time_plot_buffer):],
        average_truth[ssvep_chans_dict[c], -(HORIZON + time_plot_buffer):],
        color='k', alpha=0.4, label='Truth'
    )
    axs[i].plot(t[-HORIZON:], average_preds[ssvep_chans_dict[c], :], color='steelblue', alpha=0.8, label='Forecast')
    axs[i].fill_between(
        t[-HORIZON:],
        average_preds[ssvep_chans_dict[c], :] - preds_sem[ssvep_chans_dict[c], :],
        average_preds[ssvep_chans_dict[c], :] + preds_sem[ssvep_chans_dict[c], :],
        color='steelblue', alpha=0.2, label=r'$\pm \ 1 \text{ SEM}$'
    )
    axs[i].axvline(t[-HORIZON], color='black', linestyle=':', label='Horizon Start')
    axs[i].set_title(c)

    if i > 1:
        axs[i].set_xlabel('Time (ms)')
    if i in [0, 2]:
        axs[i].set_ylabel(r'Voltage ($\mu V$)')
    if i == 3:
        axs[i].legend(frameon=False)

plt.suptitle(f'Average S6 / Mamba Forecast\n(Horizon={HORIZON_MS} ms)')
plt.tight_layout()
plt.show()

# Summary